# 05 — Modelling
**QM640 Data Analytics Capstone | Walsh College**

This notebook implements all four statistical and machine learning models:

| Model | Research Question |
|---|---|
| Multiple Linear Regression | RQ1 — olfactory predictors of rating |
| One-Way ANOVA + Tukey HSD | RQ2 — concentration level effects |
| Two-Way ANOVA | RQ3 — gender × era cohort interaction |
| Moderated Multiple Regression | RQ4 — popularity as moderator |
| Random Forest + SHAP | Overall feature importance |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from pathlib import Path

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
FIG_DIR   = Path("../outputs/figures")
MODEL_DIR = Path("../outputs/models")
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/processed/fragrantica_features.csv")
print(f"Shape: {df.shape}")

## 5.1 Multiple Linear Regression — RQ1 (Olfactory Predictors)

In [ ]:
# Select note & accord binary columns as features
note_cols  = [c for c in df.columns if c.startswith(("top_", "heart_", "base_", "accord_"))]
X_rq1 = df[note_cols].fillna(0)
y     = df["rating_score"].dropna()
X_rq1 = X_rq1.loc[y.index]

X_train, X_test, y_train, y_test = train_test_split(X_rq1, y, test_size=0.2, random_state=42)

lr = Ridge(alpha=1.0)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f"RQ1 — Ridge Regression")
print(f"  R²   : {r2_score(y_test, y_pred):.4f}")
print(f"  RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"  MAE  : {mean_absolute_error(y_test, y_pred):.4f}")

# Cross-validation
cv_scores = cross_val_score(lr, X_rq1, y, cv=5, scoring="r2")
print(f"  5-fold CV R² : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 5.2 One-Way ANOVA + Tukey HSD — RQ2 (Concentration)

In [ ]:
conc_groups = ["Eau De Parfum", "Eau De Toilette", "Parfum", "Eau De Cologne"]
df_anova = df[df["concentration"].isin(conc_groups)][["concentration", "rating_score"]].dropna()

groups = [df_anova[df_anova["concentration"] == g]["rating_score"].values for g in conc_groups]
f_stat, p_val = stats.f_oneway(*groups)
print(f"One-Way ANOVA: F = {f_stat:.4f}, p = {p_val:.6f}")

tukey = pairwise_tukeyhsd(df_anova["rating_score"], df_anova["concentration"])
print(tukey.summary())

## 5.3 Two-Way ANOVA — RQ3 (Gender × Era)

In [ ]:
df_rq3 = df[["rating_score", "gender_label", "era_cohort"]].dropna()
df_rq3 = df_rq3[df_rq3["gender_label"].isin(["Women", "Men", "Unisex"])]

model_rq3 = smf.ols("rating_score ~ C(gender_label) + C(era_cohort) + C(gender_label):C(era_cohort)",
                    data=df_rq3).fit()
from statsmodels.stats.anova import anova_lm
anova_table = anova_lm(model_rq3, typ=2)
print(anova_table)

## 5.4 Random Forest + SHAP — Overall Feature Importance

In [ ]:
feature_cols = note_cols + ["num_votes_norm", "release_year_norm"]
X_rf = df[feature_cols].fillna(0).loc[y.index]

X_tr, X_te, y_tr, y_te = train_test_split(X_rf, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)

y_rf_pred = rf.predict(X_te)
print(f"Random Forest")
print(f"  R²   : {r2_score(y_te, y_rf_pred):.4f}")
print(f"  RMSE : {np.sqrt(mean_squared_error(y_te, y_rf_pred)):.4f}")
print(f"  MAE  : {mean_absolute_error(y_te, y_rf_pred):.4f}")

# Save model
joblib.dump(rf, MODEL_DIR / "random_forest.pkl")
print("Model saved.")

In [ ]:
# SHAP values
explainer  = shap.TreeExplainer(rf)
shap_vals  = explainer.shap_values(X_te.iloc[:500])

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_te.iloc[:500], plot_type="bar", show=False, max_display=20)
plt.title("SHAP Feature Importance (Top 20)")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()